<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 37px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  Machine Learning<br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  -<br>
  </div> 
  
<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      Illustration with Random Forests<br><br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 25px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      <font color=orange> 6 - Boosted Trees </font>
  </div>

<a id="plan"></a>

#### Regresssion with Boosted Trees

$\qquad \bullet$ [Tree Bagging vs Tree Boosting](#bagging_vs_boosting)<br>
$\qquad \bullet$ [Base model](#base_model)<br>
$\qquad \bullet$ [Learning rate](#learning_rate)<br>
$\qquad \bullet$ [Validation of the number of trees](#tree_validation)<br>
$\qquad \bullet$ [Hyperparameter tuning](#hyperparameter_tuning)<br>
$\qquad \bullet$ [Feature importance](#feature_importance)<br>

[Bottom](#bottom)

### Imports

In [ ]:
# base modules
import math
from pathlib import Path

# for manipulating data
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool, cv

# for Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

In [ ]:
from eclyon.transforms import process_df, split_vals

# Regression with Boosted Trees

[Back to top](#plan)

Tree boosting is still evolving as of today due to the difficulty of optimizing existing implementations. We list here most important implementations :

| Package | year | paper |
|-----|-----|-----|
| (class sklearn.ensemble.GradientBoostingRegressor) Scikit-learn | 1999 | [Stochastic Gradient Boosting](https://statweb.stanford.edu/~jhf/ftp/stobst.pdf)  |
| (class sklearn.ensemble.HistGradientBoostingRegressor) Scikit-learn |  |
| XGBoost | 2016 | [XGBoost: A Scalable Tree Boosting System](https://arxiv.org/pdf/1603.02754.pdf) |
| LightGBM | 2017 | [LightGBM: A Highly Efficient Gradient Boosting Decision Tree](https://papers.nips.cc/paper/2017/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf) |
| Catboost | 2018 | [CatBoost: gradient boosting with categorical features support](https://arxiv.org/pdf/1810.11363.pdf) |

Here we propose an exploration (with many plots !) of tree boosting using catboost.

### $\bullet$ Tree Bagging vs Tree Boosting

[Back to top](#plan)

Bagged Trees (e.g. Random Forests) and Boosted Trees are both models consisting of a set of decision trees. They however differs in _how the final ouput is computed_ based on the individual tree decisions, and _how these trees are optimized_, that is, what is the followed splitting strategy. Indeed, as we saw in section 3, Random forests have their output given as the _average_ of each individual tree output, and each individual tree is trained _independently_ with the same splitting procedure. This is not the case for boosted trees :

#### 1) Tree Boosting Output

The output of a boosted tree model $\mathcal{B}$ is computed as a _weighted sum_ of each individual tree output on a data point $x$ :

$$ \mathcal{B}(x) = \sum_{i = 1}^{n} \mathcal{T}_i(x) $$

#### 2) Tree Boosting splitting strategy

The learning objective of a boosted tree model is to approximate the true value $y$ of a training data point $x$ by getting each tree _trying to correct the error obtained by summing previous trees_. In other words we want, for each data point $x$, that **each of the following quantities**

$$ \sum_{i = 1}^{t}  \mathcal{T}_i(x) \qquad 1 \leqslant t \leqslant n$$

to be as close as possible from the expected value $y$. 

In other words, we expect the first tree $\mathcal{T}_1$ to predict as best as possible the expected value $y$, while we expect the second tree $\mathcal{T}_2$ to predict as best as possible _the first tree error_, so that the sum $\mathcal{T}_1 + \mathcal{T}_2$ yields a better predictor than $\mathcal{T}_1$ alone. Subsequently  we expect the third tree $\mathcal{T}_3$ to predict as best as possible the error of $\mathcal{T}_1 + \mathcal{T}_2$, so that the sum $\mathcal{T}_1 + \mathcal{T}_2 + \mathcal{T}_3$ yields a betten predictor, etc.

This overall objective does not directly drive the training procedure, but rather yields a formulation of the splitting strategy for each individual tree: Since we expect each tree to correct errors performed by previous trees, _the splitting strategy followed by each tree takes into account the ^rediction performed by previous trees_. In this sense, splittings are performed jointly, rather than for random forests where splittings are performed independently. 

The complete formulation of the splitting strategy in terms of previous tree prediction is done in [XGBoost: A Scalable Tree Boosting System](https://arxiv.org/pdf/1603.02754.pdf), Section 2.2 (see also these [docs](https://xgboost.readthedocs.io/en/latest/tutorials/model.html)).



### $\bullet$ Base model

[Back to top](#plan)

Let us import first the data, as done in the course's main notebook :

In [ ]:
path_to_repo = Path("..").resolve()
path_to_data = path_to_repo / "data"

In [ ]:
df_transformed = pd.read_feather(path_to_data / "transformed")

df, y, nas = process_df(df_transformed, "SalePrice")

In [ ]:
n_total = len(df)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train, X_valid = split_vals(df, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small, _ = split_vals(df, n_small)
y_small, _ = split_vals(y, n_small)

y_train = y_train.tolist()
y_valid = y_valid.tolist()
y_small = y_small.tolist()

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, len(y_small)))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, len(y_train)))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, len(y_valid)))

In [ ]:
def rmse(y_gold, y_pred):
    return math.sqrt(((y_gold - y_pred) ** 2).mean())


def print_score(m, X_train, y_train, X_valid, y_valid):
    print("RMSE on train set: {:.4f}".format(rmse(m.predict(X_train), y_train)))
    print("RMSE on valid set: {:.4f}".format(rmse(m.predict(X_valid), y_valid)))
    print("R^2 on train set: {:.4f}".format(m.score(X_train, y_train)))
    print("R^2 on valid set: {:.4f}".format(m.score(X_valid, y_valid)))
    if hasattr(m, "oob_score_"):
        print("R^2 on oob set: {:.4f}".format(m.oob_score_))
    return

Let us now create the model itself, with default parameters as they provide a _really_ good baseline almost all the time. Catboost has a lot of parameters available for initializing a model, and the complete list can be accessed [here](https://catboost.ai/docs/concepts/python-reference_catboostregressor.html#python-reference_catboostregressor) and their explanation [here](https://catboost.ai/docs/concepts/python-reference_parameters-list.html#python-reference_parameters-list).

In [ ]:
# help(CatBoostRegressor)

In [ ]:
model = CatBoostRegressor(
    n_estimators=500,  # iterations, n_estimators, num_boost_round, num_trees, default is 1000
    random_seed=42,
    loss_function="RMSE",
    custom_metric="R2",
    allow_writing_files=False,
    logging_level="Silent",
)

In [ ]:
model.fit(X_small, y_small)

In [ ]:
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
predictions = model.predict(X_valid.iloc[:10, :])

print(np.exp(y_valid[:10]))
print([int(np.exp(pred)) for pred in predictions])

Let us train a random forest with same number of trees and default hyperparameters in order to get the training time difference :

In [ ]:
sk_model = RandomForestRegressor(
    n_estimators=500,
    n_jobs=-1,
    random_state=42,
)
sk_model.fit(X_small, y_small)

print_score(sk_model, X_small, y_small, X_valid, y_valid)

We observe that the boosted tree overfitted noticeably less !

### $\bullet$ Learning rate

[Back to top](#plan)

The learning rate $\lambda$ is a scalar, typically between $0.0$ and $1.0$, such that during each boosting iteration $n$ the resulting tree $\mathcal{T}_n$ gets merged with the previous ensemble $\mathcal{B}_{n}$ through the rule 

$$  \mathcal{B}_{n+1} = \mathcal{B}_{n} + \lambda  \mathcal{T}_n $$

Therefore, instead of applying the full correction of $\mathcal{B}_{n}$ yield by the new tree $\mathcal{T}_n$, we only add a fraction of this correction.

The learning rate is by default computed following a certain heuristics, which takes into account the number of trees. Specifying certains parmeters replaces this heuristics by the default value $\lambda = 0.03$.

Let us try with a bigger learning rate :

In [ ]:
help(CatBoostRegressor)

In [ ]:
model = CatBoostRegressor(
    n_estimators=200,  # iterations, n_estimators, num_boost_round, num_trees
    learning_rate=1.0,  # learning_rate, eta,
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)

In [ ]:
model.fit(X_small, y_small)

In [ ]:
print_score(model, X_small, y_small, X_valid, y_valid)

Let us now try with a smaller learning rate, and more trees to alleviate :

In [ ]:
model = CatBoostRegressor(
    n_estimators=1000,   # iterations, n_estimators, num_boost_round, num_trees
    learning_rate=0.05,  # learning_rate, eta,
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)

model.fit(X_small, y_small)

print_score(model, X_small, y_small, X_valid, y_valid)

We can notice a 1% gain of $R^2$ on the validation set !

### $\bullet$ Validation of the number of trees

[Back to top](#plan)

A feature that is natively integrated into the Catboost package is the possibility to trace and plot the model scores as the number of trees grow, on both the training and a given validation set. This allows to tune the number of trees of the final model on a validation set. This feature can be seen as an _early stopping_ process focuced on the number of trees forming the model, and tunning this parameter can improve the model performances.

In [ ]:
# see the full list of hyperparameters at
# https://catboost.ai/docs/concepts/python-reference_catboostregressor.html#python-reference_catboostregressor
model = CatBoostRegressor(
    n_estimators=500,  # iterations, n_estimators, num_boost_round, num_trees
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Verbose",
)
model.fit(X_small, y_small, eval_set=(X_valid, y_valid))

print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
model.tree_count_

In [ ]:
model = CatBoostRegressor(
    n_estimators=1500,  # iterations, n_estimators, num_boost_round, num_trees
    learning_rate=0.1,  # learning_rate, eta,
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)
model.fit(X_train, y_train, eval_set=(X_valid, y_valid))

print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
model.tree_count_

In [ ]:
valid_pool = Pool(X_valid, y_valid)

r2_scores = model.eval_metrics(valid_pool, metrics="R2", plot=True)

We want to draw a _cross-validated_ RMSE score of the model, instead of scores computed on a separate validation set, as the number of tree grows :

In [ ]:
cv_dataset = Pool(data=X_small, label=y_small)

params = {
    "n_estimators": 2000,
    "loss_function": "RMSE",
    "verbose": False,
}
scores = cv(cv_dataset, params, fold_count=5, plot="True")

We can then select a number of trees that sounds natural after visual inspection. It yields an improvement of our validation score, without the use of this validation set !

In [ ]:
model = CatBoostRegressor(
    n_estimators=2000,  # iterations, n_estimators, num_boost_round, num_trees
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)

model.fit(X_small, y_small)

print_score(model, X_small, y_small, X_valid, y_valid)

### $\bullet$ Hyperparameter tuning

[Back to top](#plan)

In [ ]:
model = CatBoostRegressor(
    n_estimators=300,  # iterations, n_estimators, num_boost_round, num_trees
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)

In [ ]:
# several grids can be cast into a list
# here we provide only one grid
param_grid = [
    {
        "min_data_in_leaf": [2, 10, 50],  # equivalent of scikit-learn's 'min_samples_leaf'
        "colsample_bylevel": [0.5, 0.75],  # equivalent of scikit-learn's max_features
        "subsample": [0.25, 0.5, 1],  # equivalent of scikit-learn's max_samples
    },
]

In [ ]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=None,  # uses estimator's default score method
    refit=True,  # keep a fitted version of the overall best model
    cv=5,
    return_train_score=True,
    verbose=2,
    n_jobs=2,
)

grid.fit(X_small, y_small)

In [ ]:
best_model = grid.best_estimator_
best_model

In [ ]:
params = best_model.get_all_params()

for k, v in params.items():
    print(k, v)

In [ ]:
print_score(best_model, X_small, y_small, X_valid, y_valid)

In [ ]:
# randomized splitting strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(best_model, X_small, y_small, cv=cv, n_jobs=-1)

print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

### $\bullet$ Feature importance

[Back to top](#plan)

Catboost has a custom procedure for computing the feature importance, explained in details [here](https://catboost.ai/docs/concepts/fstr.html#fstr__regular-feature-importance). The result is a non-negative weight for each expanatory variable, with total sum equal to $1.0$

In [ ]:
model = CatBoostRegressor(
    n_estimators=300,  # n_estimators, num_boost_round, num_trees
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)

model = model.fit(X_small, y_small, plot=False)

In [ ]:
train_pool = Pool(X_train, y_train)

feature_importances = model.get_feature_importance(train_pool)
feature_names = X_train.columns
for score, name in sorted(zip(feature_importances, feature_names), reverse=True):
    print("{}: {}".format(name, score))

In [ ]:
cols = [n for n, i in zip(feature_names, feature_importances) if i > 0.005]

In [ ]:
len(cols)

In [ ]:
df_keep = df[cols]

n_total = len(df_keep)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

X_train_keep, X_valid_keep = split_vals(df_keep, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small_keep, _ = split_vals(df_keep, n_small)
y_small, _ = split_vals(y, n_small)

y_train = y_train.tolist()
y_valid = y_valid.tolist()
y_small = y_small.tolist()

print("Number of small training data points: X = {}, y = {}".format(X_small_keep.shape, len(y_small)))
print("Number of full training data points: X = {}, y = {}".format(X_train_keep.shape, len(y_train)))
print("Number of validation data points: X = {}, y = {}".format(X_valid_keep.shape, len(y_valid)))

In [ ]:
model = CatBoostRegressor(
    n_estimators=300,  # n_estimators, num_boost_round, num_trees
    random_seed=42,
    loss_function="RMSE",
    allow_writing_files=False,
    logging_level="Silent",
)

model = model.fit(X_small_keep, y_small)

print_score(model, X_small_keep, y_small, X_valid_keep, y_valid)

[Back to top](#plan)